### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# Hugging Face transformers 및 관련 라이브러리 설치
!pip install transformers grad-cam -q

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoImageProcessor, AutoModelForImageClassification
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print("✅ 라이브러리 로드 완료")

---
#### 2️⃣ 경로 설정

In [ ]:
# Google Colab 사용 시 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Google Colab 사용 시 작업 디렉토리로 이동
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-week4/4차시/06_car_damage_classification/code"

In [ ]:
# 작업 디렉토리 설정
WORK_DIR = '/content/drive/MyDrive/2026_AI_Advanced_Study-week4/4차시/06_car_damage_classification'

MODEL_PATH = os.path.join(WORK_DIR, 'runs/classification/final_model')

os.chdir(WORK_DIR)
print(f"작업 디렉토리: {WORK_DIR}")
print(f"모델 경로: {MODEL_PATH}")

# 데이터 경로 확인
DATA_DIR = Path(WORK_DIR) / 'data'
if DATA_DIR.exists():
    print(f"✅ 데이터 디렉토리 발견: {DATA_DIR}")
else:
    print(f"❌ 데이터 디렉토리를 찾을 수 없습니다: {DATA_DIR}")

---
#### 3️⃣ 모델 로드

In [ ]:
# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 및 프로세서 로드
model = AutoModelForImageClassification.from_pretrained(MODEL_PATH).to(device)
processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
model.eval()

# 클래스 이름 정의
CLASS_NAMES_EN = [
    "0_minor-dent",
    "1_minor-scratch", 
    "2_moderate-broken",
    "3_moderate-dent",
    "4_moderate-scratch",
    "5_tire_flat"
]

# 🎯 [미션] 클래스 이름 정의 (우리가 찾고 싶은 파손 종류 5가지)
CLASS_NAMES_KO = [
    "0_경미한 찌그러짐",
    ____________,
    ____________,
    ____________,
    ____________,
    ____________,
]

# 기본 언어 설정 (한국어)
CLASS_NAMES = CLASS_NAMES_KO

print(f"✅ 모델 로드 완료 (Device: {device})")
print(f"클래스: {CLASS_NAMES}")

---
#### 4️⃣ 시각화 함수 정의

In [ ]:
class ModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        return self.model(x).logits


def visualize_prediction(image_path, model, processor, class_names, class_names_ko, device):
    
    # 이미지 로드 및 전처리
    image = Image.open(image_path).convert('RGB')
    original_image = np.array(image)
    image_float = original_image.astype(np.float32) / 255.0
    
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs['pixel_values'].to(device)

    # 예측
    with torch.no_grad():
        logits = model(pixel_values).logits
        probs = F.softmax(logits, dim=-1)[0]
        pred_class = torch.argmax(probs).item()
        pred_prob = probs[pred_class].item()
    
    # Grad-CAM 생성
    try:
        wrapped_model = ModelWrapper(model)
        target_layers = [wrapped_model.model.mobilenet_v2.conv_1x1]
        targets = [ClassifierOutputTarget(pred_class)]
        
        cam = GradCAM(model=wrapped_model, target_layers=target_layers)
        grayscale_cam = cam(input_tensor=pixel_values, targets=targets)[0, :]
        
        # 224x224로 리사이즈하여 CAM 오버레이
        resized_img = cv2.resize(image_float, (224, 224))
        cam_visualization = show_cam_on_image(resized_img, grayscale_cam, use_rgb=True)
        
    except Exception as e:
        print(f"⚠️ Grad-CAM 생성 실패: {e}")
        cam_visualization = original_image
    
    # 시각화
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 1. 원본 이미지
    axes[0].imshow(original_image)
    axes[0].set_title("image", fontsize=16, fontweight='bold', pad=15)
    axes[0].axis('off')
    
    # 2. Grad-CAM
    axes[1].imshow(cam_visualization)
    axes[1].set_title(f"Grad-CAM\nPrediction: {class_names[pred_class]}", 
                      fontsize=16, fontweight='bold', color='#d62728', pad=15)
    axes[1].axis('off')
    
    # 3. 확률 분포 (클래스 순서 고정: 0→1→2→3→4→5)
    probs_np = probs.cpu().numpy()
    colors = ['#d62728' if i == pred_class else '#1f77b4' for i in range(len(class_names))]
    
    bars = axes[2].barh(range(len(class_names)), probs_np, color=colors, alpha=0.85, height=0.7)
    axes[2].set_yticks(range(len(class_names)))
    axes[2].set_yticklabels(class_names, fontsize=11)
    axes[2].set_xlabel('probabilities', fontsize=13, fontweight='bold')
    axes[2].set_title('prob per class', fontsize=16, fontweight='bold', pad=15)
    axes[2].set_xlim(0, 1.0)
    axes[2].grid(axis='x', alpha=0.3, linestyle='--')
    axes[2].invert_yaxis()  # 클래스 0이 위로
    
    # 확률 값 표시
    for i, (bar, prob) in enumerate(zip(bars, probs_np)):
        x_pos = prob + 0.02 if prob < 0.85 else prob - 0.02
        ha = 'left' if prob < 0.85 else 'right'
        color = 'black' if prob < 0.85 else 'white'
        axes[2].text(x_pos, i, f'{prob:.1%}', 
                    va='center', ha=ha, fontsize=11, fontweight='bold', color=color)
    
    plt.tight_layout()
    plt.show()
    
    # 텍스트 결과 출력
    print("\n" + "="*70)
    print(f"🎯 예측 결과: {class_names_ko[pred_class]}")
    print(f"📊 신뢰도: {pred_prob:.2%}")
    print("="*70)
    
    print("\n📈 전체 확률 분포:")
    for i, (name, prob) in enumerate(zip(class_names_ko, probs_np)):
        marker = "👉" if i == pred_class else "  "
        bar = "█" * int(prob * 30)
        print(f"{marker} {name:25s} {bar:30s} {prob:6.2%}")
    print()

print("✅ 시각화 함수 정의 완료")

---
#### 5️⃣ 단일 이미지 테스트

In [ ]:
# 🎯 [미션] 테스트 이미지 경로 설정
# 경로를 바꿔가며 다른 차들도 테스트해보세요!
TEST_IMAGE = '__________________'

if os.path.exists(TEST_IMAGE):
    print(f"📸 테스트 이미지: {TEST_IMAGE}\n")
    visualize_prediction(TEST_IMAGE, model, processor, CLASS_NAMES_EN, CLASS_NAMES_KO, device)
else:
    print(f"❌ 이미지를 찾을 수 없습니다: {TEST_IMAGE}")